# Visualization of phi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from heavyedge import ProfileData

In [ ]:
phi = pd.read_csv("phi.csv")["phi"]
with ProfileData("phi-profiles.h5") as profile_data:
    x = profile_data.x()
    Ys, Ls, _ = profile_data[:]
prob = pd.read_csv("phi-class_proba.csv")
phi_selected = pd.read_csv("phi-selected.csv")["phi"]

In [ ]:
sort_idx = np.argsort(phi_selected)
Ys = Ys[sort_idx, :]
Ls = Ls[sort_idx]
prob = prob.iloc[sort_idx, :]
phi_selected = phi_selected[sort_idx]

## Definition of phi

In [ ]:
from heavyedge_features.iproj import signed_iproj

fix, axes = plt.subplots(3, 2, figsize=(10, 9))

axes[0, 0].plot(Ys[0])
axes[0, 0].set_title("Profile")

axes[1, 0].bar(prob.columns, prob.iloc[0].values, color="tab:blue")
axes[1, 0].set_xticks(range(len(prob.columns)))
axes[1, 0].set_xticklabels(prob.columns, rotation=45, ha="right")
axes[1, 0].set_title("Class Probabilities")

axes[2, 0].bar(
    prob.columns, signed_iproj(prob.iloc[0].values, [1])[1], color="tab:orange"
)
axes[2, 0].set_xticks(range(len(prob.columns)))
axes[2, 0].set_xticklabels(prob.columns, rotation=45, ha="right")
axes[2, 0].set_title("Projected probabilities")

axes[0, 1].plot(Ys[-2])
axes[0, 1].set_title("Profile")

axes[1, 1].bar(prob.columns, prob.iloc[-2].values, color="tab:blue")
axes[1, 1].set_xticks(range(len(prob.columns)))
axes[1, 1].set_xticklabels(prob.columns, rotation=45, ha="right")
axes[1, 1].set_title("Class Probabilities")

axes[2, 1].bar(
    prob.columns, signed_iproj(prob.iloc[-2].values, [1])[1], color="tab:orange"
)
axes[2, 1].set_xticks(range(len(prob.columns)))
axes[2, 1].set_xticklabels(prob.columns, rotation=45, ha="right")
axes[2, 1].set_title("Projected probabilities")

plt.tight_layout()

## Distribution of phi

In [ ]:
counts, edges = np.histogram(phi, bins=40)

N = len(phi_selected)
colors = [plt.cm.tab10(i) for i in range(N)]

fig = plt.figure(figsize=(6, 4))
fig.set_layout_engine("none")
gs = fig.add_gridspec(
    2,
    N,
    hspace=0.35,
    height_ratios=[1, 2],
    left=0.12,
    right=0.97,
    bottom=0.12,
    top=0.88,
)

ax_profiles = [fig.add_subplot(gs[0, i]) for i in range(N)]
fig.supxlabel("Edge shapes", y=0.99, va="top")
for i, (Y, L) in enumerate(zip(Ys, Ls)):
    Y = Ys[i]
    L = Ls[i]
    Y_norm = Y[:L] / Y[:L].max()
    ax_profiles[i].plot(x[:L], Y_norm, lw=0.8, color=colors[i])
    ax_profiles[i].set_xticks([])
    ax_profiles[i].set_yticks([])
    ax_profiles[i].axis("off")

ax_bot = fig.add_subplot(gs[1, :])
ax_bot.stairs(counts, edges, fill=True, color="gray", alpha=0.7, edgecolor="white")
for i, phi in enumerate(phi_selected):
    ax_bot.axvline(phi, ls="--", color=colors[i])
ax_bot.set_xlabel(r"$\phi$")
ax_bot.spines[["top", "right"]].set_visible(False)
ax_bot.tick_params(axis="both", width=0.3)
fig.show()